In [23]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [24]:
## loading the trained model, scaler pickle, onehot

model=load_model('model.h5')

## load the encoder and scalar
with open('onehot_encoder_geo.pkl', 'rb') as file:
    onehot_encoder_geo = pickle.load(file)

with open('label_encoder_gender.pkl', 'rb') as file:
    label_encoder_gender = pickle.load(file)

with open('scaler.pkl', 'rb') as file:
    scaler = pickle.load(file)

In [25]:
# example input data

input_data = {
    'CreditScore': 700,
    'Geography': 'France',
    'Gender': 'Male',
    'Age': 35,
    'tenure': 5,
    'Balance': 10000.0,
    'NumOfProducts': 2,
    'HasCrCard': 1,
    'IsActiveMember': 1,
    'EstimatedSalary': 50000.0
}

In [26]:
# one-hot encode 'Geography'
geo_encoded = onehot_encoder_geo.transform([[input_data['Geography']]]).toarray()
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df

c:\Users\sudha\AppData\Local\Programs\Python\Python313\Lib\site-packages\sklearn\utils\validation.py:2739: UserWarning: X does not have valid feature names, but OneHotEncoder was fitted with feature names
  warnings.warn(


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [27]:
# converting it into a DataFrame

input_df=pd.DataFrame([input_data])
input_df

,CreditScore,Geography,Gender,Age,tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,700,France,Male,35,5,10000.0,2,1,1,50000.0


In [28]:
## encoding categoriccal data

input_df['Gender'] = label_encoder_gender.transform(input_df['Gender'])
input_df

,CreditScore,Geography,Gender,Age,tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary
0,700,France,1,35,5,10000.0,2,1,1,50000.0


In [29]:
# concatination with one hot encoded data

input_df=pd.concat([input_df.drop("Geography", axis=1), geo_encoded_df], axis=1)
input_df

,CreditScore,Gender,Age,tenure,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,Geography_France,Geography_Germany,Geography_Spain
0,700,1,35,5,10000.0,2,1,1,50000.0,1.0,0.0,0.0


In [33]:
# 1. Fix capitalization (rename 'tenure' to 'Tenure')
if 'tenure' in input_df.columns:
    input_df = input_df.rename(columns={'tenure': 'Tenure'})

# 2. Force input_df to match the exact columns and order expected by the scaler
# (Fills missing columns with 0 and drops extra unseen columns)
input_df = input_df.reindex(columns=scaler.feature_names_in_, fill_value=0)

# 3. Scale the input data
input_scaled = scaler.transform(input_df)
print(input_scaled)

[[ 0.50178794  0.91324755 -0.37056859 -0.00134472 -1.05836066  0.80843615
   0.64920267  0.97481699 -0.87683221 -0.57946723 -0.57638802]]


In [34]:
#predict churn

prediction= model.predict(input_scaled)
prediction


1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 55ms/step


array([[0.01214733]], dtype=float32)

In [35]:
prediction_proba = prediction[0][0]  # Assuming the model outputs a single probability for binary classification


In [36]:
prediction_proba

np.float32(0.012147331)

In [37]:
if prediction_proba > 0.5:
    print("The customer is likely to churn.")
else:
    print("The customer is not likely to churn.")

The customer is not likely to churn.
